<a href="https://colab.research.google.com/github/bintadjan/Field-scan/blob/main/semaine_1_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
base = "/content/drive/MyDrive/fieldscan"
for d in ["data/drone_zenodo", "data/roboflow", "data/bananalsd_originales",
          "data/bananalsd_augmentees", "data/sentinel2", "data/tanzanie_test",
          "notebooks", "models"]:
    os.makedirs(f"{base}/{d}", exist_ok=True)

In [ ]:
import os

# 1. Créer le dossier src/ s'il n'existe pas
os.makedirs("src", exist_ok=True)
print("Dossier 'src/' créé avec succès !")

# 2. Créer le fichier requirements.txt
requirements_content = """rasterio>=1.3.0
geopandas>=0.12.0
shapely>=2.0.0
torch>=2.0.0
torchvision>=0.15.0
matplotlib>=3.5.0
numpy>=1.20.0
pandas>=1.4.0
scipy>=1.8.0
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)
print("Fichier 'requirements.txt' créé avec succès !")

# 3. Créer le fichier data_card.md
datacard_content = """# Carnet de Données - Projet Field Scan

## 1. Imagerie Satellitaire (Sentinel-2)
* **Source :** Copernicus Browser / ESA
* **Licence :** Open Data Copernicus (CC BY 4.0)
* **Résolution :** 10 mètres par pixel (bandes RVB et NIR)
* **Date d'acquisition :** Mars 2026 (Saison des pluies) & Juin-Août 2026 (Saison sèche)
* **Projection / CRS :** EPSG:32717
* **Biais et limites :** Présence de nuages sur certaines tuiles (filtrage nécessaire) ; résolution de 10m insuffisante pour compter les pieds individuels (réservée à l'analyse de vigueur/stress global).

## 2. Dataset Maladies Foliaires (Tanzanie & BananaLSD)
* **Source :** Harvard Dataverse (Banana Leaves Imagery Dataset) & BananaLSD
* **Licence :** Creative Commons (CC BY)
* **Composition :** ~5 600 images saines et ~5 700 images de Black Sigatoka (Tanzanie) + dataset BananaLSD (cordana, pestalotiopsis, etc.).
* **Résolution / Type :** Photos de terrain prises de près (smartphone/appareil photo au sol).
* **Biais et limites (Domain Gap) :** Écart important entre ces images prises au sol et de près, et les vues aériennes de drone ou satellite. Le modèle devra être utilisé comme un outil d'aide au diagnostic local.
"""

with open("data_card.md", "w", encoding="utf-8") as f:
    f.write(datacard_content)
print("Fichier 'data_card.md' créé avec succès !")

In [ ]:
print("Bonjour Field Scan")

In [ ]:
!pip install rasterio matplotlib

In [ ]:
import requests

doi = "doi:10.7910/DVN/LQUWXW"
url = f"https://dataverse.harvard.edu/api/datasets/:persistentId/versions/:latest/files?persistentId={doi}"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)

print("Code :", r.status_code)
print("Type :", r.headers.get("Content-Type"))
print(r.text[:500])

In [ ]:
import requests

doi = "doi:10.7910/DVN/LQUWXW"
url = f"https://dataverse.harvard.edu/api/datasets/:persistentId/versions/:latest/files?persistentId={doi}"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)

for f in r.json()["data"]:
    d = f["dataFile"]
    print(d["id"], "|", d["filename"], "|", round(d["filesize"]/1e6, 1), "Mo")

In [ ]:
!wget --user-agent="Mozilla/5.0" -O "/content/drive/MyDrive/fieldscan/data/tanzanie_test/Bananas_TZ_data_sheet.pdf" "https://dataverse.harvard.edu/api/access/datafile/6420462"
!ls -lh /content/drive/MyDrive/fieldscan/data/tanzanie_test/

In [ ]:
!wget --user-agent="Mozilla/5.0" -O /content/fichier.zip "https://dataverse.harvard.edu/api/access/datafile/ID_DU_FICHIER"
!ls -lh /content/fichier.zip

In [ ]:
!ls -lh /content/drive/MyDrive/fieldscan/data/drone_zenodo/

In [ ]:
!rm "/content/drive/MyDrive/fieldscan/data/drone_zenodo/archive.zip"

In [ ]:
import requests

doi = "doi:10.7910/DVN/LQUWXW"
url = f"https://dataverse.harvard.edu/api/datasets/:persistentId/versions/:latest/files?persistentId={doi}"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)

for f in r.json()["data"]:
    d = f["dataFile"]
    print(d["id"], "|", d["filename"], "|", round(d["filesize"]/1e6, 1), "Mo | md5:", d["md5"])

In [ ]:
import requests

doi = "doi:10.7910/DVN/LQUWXW"
url = f"https://dataverse.harvard.edu/api/datasets/:persistentId/versions/:latest/files?persistentId={doi}"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)

for f in r.json()["data"]:
    d = f["dataFile"]
    print(d["id"], "|", d["filename"], "|", round(d["filesize"]/1e6, 1), "Mo | md5:", d["md5"])

In [ ]:
import hashlib, os

fichiers = {
    "HEALTHY_1":        (7077478, "1b1526a893ae29385bf14b312f8893bf"),
    "BLACK_SIGATOKA_1": (7071415, "a801528927f604eb43ffdba4dbb38dfa"),
}

os.makedirs("/content/tanzanie", exist_ok=True)

for nom, (fid, md5_attendu) in fichiers.items():
    chemin = f"/content/tanzanie/{nom}.zip"
    !wget -c --user-agent="Mozilla/5.0" -q -O "{chemin}" "https://dataverse.harvard.edu/api/access/datafile/{fid}"
    h = hashlib.md5()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1<<20), b""):
            h.update(bloc)
    print(nom, "OK" if h.hexdigest() == md5_attendu else "MD5 DIFFERENT, à retélécharger")

In [ ]:
!unzip -q /content/tanzanie/HEALTHY_1.zip -d /content/tanzanie/HEALTHY_1
!unzip -q /content/tanzanie/BLACK_SIGATOKA_1.zip -d /content/tanzanie/BLACK_SIGATOKA_1

!find /content/tanzanie/HEALTHY_1 -type f | wc -l
!find /content/tanzanie/BLACK_SIGATOKA_1 -type f | wc -l
!find /content/tanzanie/HEALTHY_1 -type f | head -5

In [ ]:
from PIL import Image
import glob, random

for classe in ["HEALTHY_1", "BLACK_SIGATOKA_1"]:
    chemins = glob.glob(f"/content/tanzanie/{classe}/**/*.jpg", recursive=True)
    echantillon = random.sample(chemins, 5)
    print(classe, len(chemins), "images")
    for c in echantillon:
        print("  ", c.split("/")[-1], Image.open(c).size)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, classe in enumerate(["HEALTHY_1", "BLACK_SIGATOKA_1"]):
    chemins = glob.glob(f"/content/tanzanie/{classe}/**/*.jpg", recursive=True)
    for j, c in enumerate(random.sample(chemins, 4)):
        axes[i, j].imshow(Image.open(c))
        axes[i, j].set_title(classe)
        axes[i, j].axis("off")
plt.show()

In [ ]:
!pip install -q imagehash
import imagehash, collections

hachages = collections.defaultdict(list)
for classe in ["HEALTHY_1", "BLACK_SIGATOKA_1"]:
    for c in glob.glob(f"/content/tanzanie/{classe}/**/*.jpg", recursive=True):
        h = str(imagehash.phash(Image.open(c)))
        hachages[h].append(c)

doublons = {h: v for h, v in hachages.items() if len(v) > 1}
print(len(doublons), "groupes de doublons ou quasi-doublons")

In [ ]:
import numpy as np, pandas as pd, glob, random
from PIL import Image

lignes = []
for classe in ["HEALTHY_1", "BLACK_SIGATOKA_1"]:
    chemins = glob.glob(f"/content/tanzanie/{classe}/**/*.jpg", recursive=True)
    for c in random.sample(chemins, 300):
        img = Image.open(c)
        l, h = img.size
        petite = np.asarray(img.convert("RGB").resize((64, 64)), dtype=float)
        lignes.append({"classe": classe, "largeur": l, "hauteur": h,
                       "ratio": round(l/h, 2),
                       "R": petite[..., 0].mean(), "G": petite[..., 1].mean(),
                       "B": petite[..., 2].mean()})

df = pd.DataFrame(lignes)
print(df.groupby("classe")[["largeur", "hauteur", "ratio", "R", "G", "B"]].describe().T.round(1).to_string())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X = df[["largeur", "hauteur", "ratio", "R", "G", "B"]]
y = (df["classe"] == "BLACK_SIGATOKA_1").astype(int)
scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0), X, y, cv=5)
print("Précision sans regarder la feuille :", scores.mean().round(2))

# Et avec la taille seulement
scores_t = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0),
                           df[["largeur", "hauteur", "ratio"]], y, cv=5)
print("Taille seule :", scores_t.mean().round(2))

In [ ]:
import hashlib, os

fichiers = {
    "HEALTHY_2":        (7077403, "d7a130566e04511c94e1e7db0e02b2a2"),
    "BLACK_SIGATOKA_2": (6078711, "b51c8e45255c440f3c4e3328d215cc25"),
}

os.makedirs("/content/tanzanie", exist_ok=True)

for nom, (fid, md5_attendu) in fichiers.items():
    chemin = f"/content/tanzanie/{nom}.zip"
    !wget -c --user-agent="Mozilla/5.0" -q -O "{chemin}" "https://dataverse.harvard.edu/api/access/datafile/{fid}"
    h = hashlib.md5()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1<<20), b""):
            h.update(bloc)
    print(nom, "OK" if h.hexdigest() == md5_attendu else "MD5 DIFFERENT, à retélécharger")

In [ ]:
!unzip -q /content/tanzanie/HEALTHY_2.zip -d /content/tanzanie/HEALTHY_2
!unzip -q /content/tanzanie/BLACK_SIGATOKA_2.zip -d /content/tanzanie/BLACK_SIGATOKA_2

for classe in ["HEALTHY_1", "HEALTHY_2", "BLACK_SIGATOKA_1", "BLACK_SIGATOKA_2"]:
    print(classe)
    !find /content/tanzanie/{classe} -type f | wc -l

In [ ]:
import glob, os

for classe in ["HEALTHY", "BLACK_SIGATOKA"]:
    dest = f"/content/tanzanie_all/{classe}"
    os.makedirs(dest, exist_ok=True)
    n = 0
    for partie in ["1", "2"]:
        for p in glob.glob(f"/content/tanzanie/{classe}_{partie}/**/*.jpg", recursive=True):
            os.symlink(p, f"{dest}/{partie}_{os.path.basename(p)}")
            n += 1
    print(classe, n, "images")

In [ ]:
import imagehash, collections
from PIL import Image

hachages = collections.defaultdict(list)
for classe in ["HEALTHY", "BLACK_SIGATOKA"]:
    for c in glob.glob(f"/content/tanzanie_all/{classe}/*.jpg"):
        hachages[str(imagehash.phash(Image.open(c)))].append(c)

doublons = {h: v for h, v in hachages.items() if len(v) > 1}
print(len(doublons), "groupes de doublons exacts")

In [ ]:
import glob, os, hashlib

for classe in ["HEALTHY", "BLACK_SIGATOKA"]:
    for partie in ["1", "2"]:
        n = len(glob.glob(f"/content/tanzanie/{classe}_{partie}/**/*.jpg", recursive=True))
        print(f"{classe}_{partie} : {n} images")
    print(f"{classe} total :", len(glob.glob(f"/content/tanzanie_all/{classe}/*.jpg")))

!du -sh /content/tanzanie/HEALTHY_1 /content/tanzanie/HEALTHY_2 /content/tanzanie/BLACK_SIGATOKA_1 /content/tanzanie/BLACK_SIGATOKA_2

attendus = {"HEALTHY_2": "d7a130566e04511c94e1e7db0e02b2a2",
            "BLACK_SIGATOKA_2": "b51c8e45255c440f3c4e3328d215cc25"}
for nom, md5 in attendus.items():
    h = hashlib.md5()
    with open(f"/content/tanzanie/{nom}.zip", "rb") as f:
        for bloc in iter(lambda: f.read(1<<20), b""):
            h.update(bloc)
    print(nom, "MD5 OK" if h.hexdigest() == md5 else "MD5 DIFFERENT")

In [ ]:
import glob, os, pandas as pd
from PIL import Image

os.makedirs("/content/tanzanie_512", exist_ok=True)
lignes = []

for classe in ["HEALTHY", "BLACK_SIGATOKA"]:
    os.makedirs(f"/content/tanzanie_512/{classe}", exist_ok=True)
    for chemin in glob.glob(f"/content/tanzanie_all/{classe}/*.jpg"):
        nom = os.path.basename(chemin)
        img = Image.open(chemin).convert("RGB")
        lignes.append({"classe": classe, "fichier": nom,
                       "largeur_orig": img.width, "hauteur_orig": img.height})
        img.thumbnail((512, 512))
        img.save(f"/content/tanzanie_512/{classe}/{nom}", quality=90)

pd.DataFrame(lignes).to_csv("/content/tanzanie_512/manifeste.csv", index=False)
print(len(lignes), "images traitées")

In [ ]:
!cd /content && zip -qr tanzanie_512.zip tanzanie_512
!ls -lh /content/tanzanie_512.zip
!cp /content/tanzanie_512.zip /content/drive/MyDrive/fieldscan/data/tanzanie_test/
!ls -lh /content/drive/MyDrive/fieldscan/data/tanzanie_test/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/fieldscan/data/bananalsd_originales/NOM_DU_ZIP.zip" -d /content/bananalsd
!find /content/bananalsd -maxdepth 3 -type d
!find /content/bananalsd -type f | wc -l

In [ ]:
!ls -lh /content/drive/MyDrive/fieldscan/data/bananalsd_originales/

In [ ]:
!unzip -q "/content/drive/MyDrive/fieldscan/data/bananalsd_originales/LE_VRAI_NOM.zip" -d /content/bananalsd

In [ ]:
!ls -lh /content/drive/MyDrive/fieldscan/data/bananalsd_originales/

In [ ]:
import glob
zips = glob.glob("/content/drive/MyDrive/fieldscan/data/bananalsd_originales/*.zip")
print(zips)
!unzip -q "{zips[0]}" -d /content/bananalsd
!find /content/bananalsd -maxdepth 4 -type d
!find /content/bananalsd -type f | wc -l

In [ ]:
import os
from collections import Counter

compte = Counter()
for racine, _, fichiers in os.walk("/content/bananalsd"):
    for f in fichiers:
        compte[(racine.replace("/content/bananalsd/", ""), os.path.splitext(f)[1].lower())] += 1

for (dossier, ext), n in sorted(compte.items()):
    print(n, "|", ext, "|", dossier)

In [ ]:
for racine, _, fichiers in os.walk("/content/bananalsd"):
    if fichiers:
        print(racine.replace("/content/bananalsd/", ""))
        for f in sorted(fichiers)[:4]:
            print("   ", f)

In [ ]:
import glob, os, shutil
from collections import Counter

racine = "/content/bananalsd"
if not os.path.exists(racine):
    zips = glob.glob("/content/drive/MyDrive/fieldscan/data/bananalsd_originales/*.zip")
    !unzip -q "{zips[0]}" -d /content/bananalsd

images = [p for p in glob.glob(f"{racine}/**/*", recursive=True)
          if p.lower().endswith((".jpg", ".jpeg", ".png"))]

def type_ensemble(p):
    chemin = p.lower()
    if "augment" in chemin:
        return "augmentee"
    return "originale"

compte = Counter(type_ensemble(p) for p in images)
print(compte)

In [ ]:
def classe(p):
    return os.path.basename(os.path.dirname(p))

print(Counter((type_ensemble(p), classe(p)) for p in images))

In [ ]:
import pandas as pd

lignes = []
for p in images:
    ens = type_ensemble(p)
    cl = classe(p)
    dossier = "bananalsd_originales" if ens == "originale" else "bananalsd_augmentees"
    dest_dir = f"/content/bananalsd_tri/{dossier}/{cl}"
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(p, f"{dest_dir}/{os.path.basename(p)}")
    lignes.append({"ensemble": ens, "classe": cl, "fichier": os.path.basename(p)})

pd.DataFrame(lignes).to_csv("/content/bananalsd_tri/manifeste.csv", index=False)
for d in ["bananalsd_originales", "bananalsd_augmentees"]:
    n = len(glob.glob(f"/content/bananalsd_tri/{d}/**/*.*", recursive=True))
    print(d, n, "fichiers")

In [ ]:
!cd /content/bananalsd_tri && zip -qr originales.zip bananalsd_originales
!cd /content/bananalsd_tri && zip -qr augmentees.zip bananalsd_augmentees
!cp /content/bananalsd_tri/originales.zip /content/drive/MyDrive/fieldscan/data/bananalsd_originales/
!cp /content/bananalsd_tri/augmentees.zip /content/drive/MyDrive/fieldscan/data/bananalsd_augmentees/
!cp /content/bananalsd_tri/manifeste.csv /content/drive/MyDrive/fieldscan/data/bananalsd_originales/
!ls -lh /content/drive/MyDrive/fieldscan/data/bananalsd_originales/ /content/drive/MyDrive/fieldscan/data/bananalsd_augmentees/

In [ ]:
!pip install -q pystac-client planetary-computer rasterio

import planetary_computer, pystac_client

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace)

bbox = [-79.86, -3.32, -79.78, -3.24]   # ouest, sud, est, nord

def chercher(debut, fin):
    s = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=f"{debut}/{fin}",
                       query={"eo:cloud_cover": {"lt": 40}})
    items = sorted(s.items(), key=lambda i: i.properties["eo:cloud_cover"])
    for i in items[:5]:
        print(i.id, "|", i.datetime.date(), "|", round(i.properties["eo:cloud_cover"], 1), "% de nuages")
    return items

print("SAISON SÈCHE (juillet à mi-octobre 2026)")
seche = chercher("2026-07-01", "2026-10-15")
print("SAISON DES PLUIES (mi-janvier à avril 2026)")
pluies = chercher("2026-01-15", "2026-04-30")

In [ ]:
import rasterio, numpy as np
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

def chercher(debut, fin, max_nuages):
    s = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=f"{debut}/{fin}",
                       query={"eo:cloud_cover": {"lt": max_nuages}})
    return sorted(s.items(), key=lambda i: i.properties["eo:cloud_cover"])

def lire(item, bande):
    with rasterio.open(item.assets[bande].href) as src:
        b = transform_bounds("EPSG:4326", src.crs, *bbox)
        fen = from_bounds(*b, transform=src.transform)
        data = src.read(1, window=fen).astype("float32")
        return data, src.window_transform(fen), src.crs

def nuages_locaux(item):
    scl, _, _ = lire(item, "SCL")
    return float(np.isin(scl, [3, 8, 9, 10]).mean() * 100)   # ombres + nuages + cirrus

seche = chercher("2026-06-01", "2026-10-15", 90)
print(len(seche), "images trouvées (saison sèche)")
for i in seche[:6]:
    print(i.id, "|", i.datetime.date(), "| tuile :", round(i.properties["eo:cloud_cover"]), "% | zone :", round(nuages_locaux(i)), "%")

print("---")
pluies = chercher("2026-01-15", "2026-04-30", 40)
for i in pluies[:5]:
    print(i.id, "|", i.datetime.date(), "| tuile :", round(i.properties["eo:cloud_cover"]), "% | zone :", round(nuages_locaux(i)), "%")

In [ ]:
import matplotlib.pyplot as plt

item = pluies[0]    # remplace 0 par l'indice de la meilleure image selon la colonne « zone »
rouge, tr, crs = lire(item, "B04")
pir, _, _ = lire(item, "B08")

rouge = (rouge - 1000) / 10000    # décalage valable pour les produits récents (à vérifier)
pir = (pir - 1000) / 10000
ndvi = (pir - rouge) / (pir + rouge + 1e-6)

print("Image :", item.id, "| taille :", ndvi.shape, "| CRS :", crs)
print("NDVI min / moyen / max :", float(np.nanmin(ndvi)), float(np.nanmean(ndvi)), float(np.nanmax(ndvi)))

plt.figure(figsize=(8, 8))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
plt.colorbar(label="NDVI")
plt.title(f"NDVI - {item.datetime.date()}")
plt.show()

In [ ]:
!pip install -q pystac-client planetary-computer rasterio

import time, numpy as np, matplotlib.pyplot as plt, rasterio
import planetary_computer, pystac_client
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

bbox = [-79.86, -3.32, -79.78, -3.24]

catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

def chercher(debut, fin, max_nuages):
    s = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox,
                       datetime=f"{debut}/{fin}",
                       query={"eo:cloud_cover": {"lt": max_nuages}})
    return sorted(s.items(), key=lambda i: i.properties["eo:cloud_cover"])

def lire(item, bande, forme=None):
    for essai in range(3):
        try:
            signe = planetary_computer.sign(item)      # lien d'accès renouvelé
            with rasterio.open(signe.assets[bande].href) as src:
                b = transform_bounds("EPSG:4326", src.crs, *bbox)
                fen = from_bounds(*b, transform=src.transform)
                kw = {"out_shape": forme, "resampling": Resampling.nearest} if forme else {}
                data = src.read(1, window=fen, **kw).astype("float32")
                return data, src.window_transform(fen), src.crs
        except Exception as e:
            if essai == 2:
                raise
            time.sleep(3)

def nuages_locaux(item):
    scl, _, _ = lire(item, "SCL")
    return float(np.isin(scl, [3, 8, 9, 10]).mean() * 100)

def ndvi_masque(item):
    rouge, tr, crs = lire(item, "B04")
    pir, _, _ = lire(item, "B08")
    scl, _, _ = lire(item, "SCL", forme=rouge.shape)
    rouge = (rouge - 1000) / 10000
    pir = (pir - 1000) / 10000
    ndvi = (pir - rouge) / (pir + rouge + 1e-6)
    valide = np.isin(scl, [4, 5])
    ndvi[~valide] = np.nan
    return ndvi, tr, crs, valide.mean() * 100

# Saison sèche : l'image du 8 août 2026
item_sec = chercher("2026-08-08", "2026-08-08", 100)[0]
print("Image sèche trouvée :", item_sec.id)

# Saison des pluies : on mesure les nuages image par image, en ignorant celles qui échouent
candidats = chercher("2026-01-15", "2026-05-15", 70)[:10]
mesures = []
for i in candidats:
    try:
        mesures.append((nuages_locaux(i), i))
    except Exception as e:
        print("Image ignorée (refusée par le serveur) :", i.id)
mesures.sort(key=lambda t: t[0])
for n, i in mesures[:5]:
    print(i.datetime.date(), "| nuages sur la zone :", round(n), "%")
item_pluie = mesures[0][1]

ndvi_s, tr, crs, ok_s = ndvi_masque(item_sec)
ndvi_p, _, _, ok_p = ndvi_masque(item_pluie)
print("SÈCHE  :", item_sec.datetime.date(), "| pixels valides :", round(ok_s), "% | NDVI moyen :", round(float(np.nanmean(ndvi_s)), 2))
print("PLUIES :", item_pluie.datetime.date(), "| pixels valides :", round(ok_p), "% | NDVI moyen :", round(float(np.nanmean(ndvi_p)), 2))

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, img, titre in [(ax[0], ndvi_s, f"Sèche {item_sec.datetime.date()}"),
                      (ax[1], ndvi_p, f"Pluies {item_pluie.datetime.date()}")]:
    m = a.imshow(img, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    a.set_title(titre)
    plt.colorbar(m, ax=a, fraction=0.046)
m = ax[2].imshow(ndvi_p - ndvi_s, cmap="coolwarm", vmin=-0.4, vmax=0.4)
ax[2].set_title("Pluies moins sèche")
plt.colorbar(m, ax=ax[2], fraction=0.046)
plt.show()

In [ ]:
scl_s, _, _ = lire(item_sec, "SCL")
scl_p, _, _ = lire(item_pluie, "SCL")

noms = {0:"sans donnée", 1:"saturé", 2:"zones sombres", 3:"ombre de nuage", 4:"végétation",
        5:"sol nu", 6:"eau", 7:"non classé", 8:"nuage moyen", 9:"nuage dense",
        10:"cirrus", 11:"neige"}

for titre, scl in [("SÈCHE", scl_s), ("PLUIES", scl_p)]:
    print(titre)
    valeurs, nb = np.unique(scl, return_counts=True)
    for v, n in zip(valeurs, nb):
        print(f"  {int(v):2d} {noms.get(int(v), '?'):15s} {n / scl.size * 100:5.1f} %")

# Comparaison sur les pixels valides des deux dates seulement
commun = ~np.isnan(ndvi_s) & ~np.isnan(ndvi_p)
print("\nPixels valides dans les deux saisons :", round(commun.mean() * 100), "%")
print("NDVI moyen sèche  (pixels communs) :", round(float(ndvi_s[commun].mean()), 3))
print("NDVI moyen pluies (pixels communs) :", round(float(ndvi_p[commun].mean()), 3))

In [ ]:
def pixels_invalides(item):
    scl, _, _ = lire(item, "SCL")
    return float(np.isin(scl, [0, 3, 8, 9, 10]).mean() * 100)   # sans donnée + nuages + ombres

candidats = chercher("2026-01-15", "2026-05-15", 70)[:15]
mesures = []
for i in candidats:
    try:
        mesures.append((pixels_invalides(i), i))
    except Exception:
        print("Image ignorée :", i.id)
mesures.sort(key=lambda t: t[0])
for n, i in mesures[:6]:
    print(i.datetime.date(), "|", i.id[-22:], "| pixels inutilisables :", round(n), "%")

item_pluie = mesures[0][1]
ndvi_p, _, _, ok_p = ndvi_masque(item_pluie)
commun = ~np.isnan(ndvi_s) & ~np.isnan(ndvi_p)
print("\nImage retenue :", item_pluie.datetime.date(), "| pixels valides :", round(ok_p), "%")
print("Pixels communs aux deux saisons :", round(commun.mean() * 100), "%")
print("NDVI moyen sèche  :", round(float(ndvi_s[commun].mean()), 3))
print("NDVI moyen pluies :", round(float(ndvi_p[commun].mean()), 3))

In [ ]:
import os, rasterio

dossier = "/content/drive/MyDrive/fieldscan/outputs"
os.makedirs(dossier, exist_ok=True)

ndvi_s, tr_s, crs_s, _ = ndvi_masque(item_sec)
ndvi_p, tr_p, crs_p, _ = ndvi_masque(item_pluie)
print("Mêmes dimensions :", ndvi_s.shape == ndvi_p.shape, "| mêmes coordonnées :", tr_s == tr_p)

def sauver(carte, tr, crs, nom):
    chemin = f"{dossier}/{nom}"
    with rasterio.open(chemin, "w", driver="GTiff", height=carte.shape[0], width=carte.shape[1],
                       count=1, dtype="float32", crs=crs, transform=tr, nodata=np.nan) as dst:
        dst.write(carte.astype("float32"), 1)
    print("Sauvegardé :", chemin)

sauver(ndvi_s, tr_s, crs_s, "ndvi_seche_2026-08-08.tif")
sauver(ndvi_p, tr_p, crs_p, "ndvi_pluies_2026-04-10.tif")
if ndvi_s.shape == ndvi_p.shape:
    sauver(ndvi_p - ndvi_s, tr_s, crs_s, "ndvi_difference_pluies_moins_seche.tif")

In [ ]:
def ndmi_masque(item):
    nir, tr, crs = lire(item, "B8A")
    swir, _, _ = lire(item, "B11", forme=nir.shape)
    scl, _, _ = lire(item, "SCL", forme=nir.shape)
    nir = (nir - 1000) / 10000
    swir = (swir - 1000) / 10000
    ndmi = (nir - swir) / (nir + swir + 1e-6)
    valide = np.isin(scl, [4, 5])
    ndmi[~valide] = np.nan
    return ndmi, tr, crs, valide.mean() * 100

ndmi_s, tr20, crs20, ok_s = ndmi_masque(item_sec)
ndmi_p, tr20p, _, ok_p = ndmi_masque(item_pluie)
print("Dimensions :", ndmi_s.shape, ndmi_p.shape, "| mêmes coordonnées :", tr20 == tr20p)

commun = ~np.isnan(ndmi_s) & ~np.isnan(ndmi_p)
print("Pixels valides dans les deux saisons :", round(commun.mean() * 100), "%")
print("NDMI moyen sèche  :", round(float(ndmi_s[commun].mean()), 3))
print("NDMI moyen pluies :", round(float(ndmi_p[commun].mean()), 3))

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, img, titre in [(ax[0], ndmi_s, f"NDMI sèche {item_sec.datetime.date()}"),
                      (ax[1], ndmi_p, f"NDMI pluies {item_pluie.datetime.date()}")]:
    m = a.imshow(img, cmap="BrBG", vmin=-0.2, vmax=0.7)
    a.set_title(titre)
    plt.colorbar(m, ax=a, fraction=0.046)
m = ax[2].imshow(ndmi_p - ndmi_s, cmap="coolwarm", vmin=-0.3, vmax=0.3)
ax[2].set_title("Pluies moins sèche")
plt.colorbar(m, ax=ax[2], fraction=0.046)
plt.show()

In [ ]:
sauver(ndmi_s, tr20, crs20, "ndmi_seche_2026-08-08.tif")
sauver(ndmi_p, tr20, crs20, "ndmi_pluies_2026-04-10.tif")
sauver(ndmi_p - ndmi_s, tr20, crs20, "ndmi_difference_pluies_moins_seche.tif")

plt.savefig("/content/drive/MyDrive/fieldscan/outputs/ndmi_cartes.png", dpi=120, bbox_inches="tight")

In [ ]:
import os
print("Drive monté :", os.path.ismount("/content/drive"))
local = "/content/drive/MyDrive/fieldscan/outputs"
print("Dossier outputs présent :", os.path.exists(local))
if os.path.exists(local):
    for f in sorted(os.listdir(local)):
        print(" -", f)

In [ ]:
import os, shutil
from google.colab import drive

local = "/content/drive/MyDrive/fieldscan/outputs"
shutil.copytree(local, "/content/outputs_sauvegarde")   # copie de sécurité hors du faux dossier

shutil.rmtree("/content/drive")                         # supprime le faux dossier Drive
drive.mount("/content/drive")                           # une fenêtre s'ouvre : autorise l'accès

dest = "/content/drive/MyDrive/fieldscan/outputs"
os.makedirs(dest, exist_ok=True)
for f in os.listdir("/content/outputs_sauvegarde"):
    shutil.copy(f"/content/outputs_sauvegarde/{f}", f"{dest}/{f}")

print("Drive monté :", os.path.ismount("/content/drive"))
for f in sorted(os.listdir(dest)):
    print(" -", f, "|", round(os.path.getsize(f"{dest}/{f}") / 1e6, 2), "Mo")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
dest = "/content/drive/MyDrive/fieldscan/outputs"
print("Drive monté :", os.path.ismount("/content/drive"))
os.makedirs(dest, exist_ok=True)
print("Fichiers déjà présents :", sorted(os.listdir(dest)))

In [ ]:
import os, time
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
dest = "/content/drive/MyDrive/fieldscan/outputs"
os.makedirs(dest, exist_ok=True)

!pip install -q pystac-client planetary-computer rasterio

import numpy as np, rasterio
import planetary_computer, pystac_client
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

t0 = time.time()
def top(msg):
    print(f"[{round(time.time() - t0)} s] {msg}", flush=True)

# Limites de temps et de nouvelles tentatives pour les lectures à distance
GDAL = dict(GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="2", GDAL_HTTP_TIMEOUT="30",
            GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif")

bbox = [-79.86, -3.32, -79.78, -3.24]
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

def trouver(jour, fin_id):
    s = catalog.search(collections=["sentinel-2-l2a"], bbox=bbox, datetime=f"{jour}/{jour}")
    for item in s.items():
        if item.id.endswith(fin_id):
            return item
    raise ValueError("Image introuvable : " + fin_id)

item_sec = trouver("2026-08-08", "T17MPS_20260808T212312")
item_pluie = trouver("2026-04-10", "T17MPS_20260410T204104")
top("Sèche : " + item_sec.id)
top("Pluies : " + item_pluie.id)

def lire(item, bande, forme=None):
    for essai in range(3):
        try:
            signe = planetary_computer.sign(item)
            with rasterio.Env(**GDAL):
                with rasterio.open(signe.assets[bande].href) as src:
                    b = transform_bounds("EPSG:4326", src.crs, *bbox)
                    fen = from_bounds(*b, transform=src.transform)
                    kw = {"out_shape": forme, "resampling": Resampling.nearest} if forme else {}
                    data = src.read(1, window=fen, **kw).astype("float32")
                    return data, src.window_transform(fen), src.crs
        except Exception as e:
            top(f"  lecture {bande} : essai {essai + 1} échoué")
            if essai == 2:
                raise
            time.sleep(3)

def indice(item, bande_a, bande_b):
    a, tr, crs = lire(item, bande_a)
    b, _, _ = lire(item, bande_b, forme=a.shape)
    scl, _, _ = lire(item, "SCL", forme=a.shape)
    a = (a - 1000) / 10000
    b = (b - 1000) / 10000
    res = (a - b) / (a + b + 1e-6)
    res[~np.isin(scl, [4, 5])] = np.nan
    return res, tr, crs

def sauver(carte, tr, crs, chemin):
    with rasterio.open(chemin, "w", driver="GTiff", height=carte.shape[0], width=carte.shape[1],
                       count=1, dtype="float32", crs=crs, transform=tr, nodata=np.nan) as dst:
        dst.write(carte.astype("float32"), 1)

# NDVI = (B08 - B04) / (B08 + B04) ; NDMI = (B8A - B11) / (B8A + B11)
travaux = [("ndvi_seche_2026-08-08.tif", item_sec, "B08", "B04"),
           ("ndvi_pluies_2026-04-10.tif", item_pluie, "B08", "B04"),
           ("ndmi_seche_2026-08-08.tif", item_sec, "B8A", "B11"),
           ("ndmi_pluies_2026-04-10.tif", item_pluie, "B8A", "B11")]

for nom, item, a, b in travaux:
    chemin = f"{dest}/{nom}"
    if os.path.exists(chemin):
        top(f"déjà présent, ignoré : {nom}")
        continue
    try:
        top(f"calcul de {nom}")
        carte, tr, crs = indice(item, a, b)
        sauver(carte, tr, crs, chemin)
        top(f"sauvegardé : {nom} | moyenne {round(float(np.nanmean(carte)), 3)}")
    except Exception as e:
        top(f"ÉCHEC pour {nom} : {type(e).__name__}")

top("Fichiers sur le Drive : " + str(sorted(os.listdir(dest))))

In [ ]:
import os, json, numpy as np, rasterio
from google.colab import drive
from rasterio.warp import transform_geom

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

dossier = "/content/drive/MyDrive/fieldscan/outputs"
with rasterio.open(f"{dossier}/ndvi_seche_2026-08-08.tif") as s:
    ndvi = s.read(1)
    tr = s.transform
    crs = s.crs

h, w = ndvi.shape
pas = 100
features = []
for r in range(0, h - pas + 1, pas):
    for c in range(0, w - pas + 1, pas):
        bloc = ndvi[r:r + pas, c:c + pas]
        valide = float(np.mean(~np.isnan(bloc)) * 100)
        if valide < 70:
            continue
        x0, y0 = tr * (c, r)
        x1, y1 = tr * (c + pas, r + pas)
        poly = {"type": "Polygon", "coordinates": [[(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]]}
        features.append({"type": "Feature",
                         "geometry": transform_geom(crs, "EPSG:4326", poly),
                         "properties": {"nom": f"G{r // pas}_{c // pas}",
                                        "ndvi_moyen": round(float(np.nanmean(bloc)), 3),
                                        "pixels_valides_pct": round(valide)}})

sortie = f"{dossier}/parcelles_grille.geojson"
with open(sortie, "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)
print(len(features), "carrés gardés | fichier :", sortie)

In [ ]:
import os, json, numpy as np, pandas as pd, rasterio
from rasterio.features import geometry_mask
from rasterio.warp import transform_geom
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
dossier = "/content/drive/MyDrive/fieldscan/outputs"

grille = json.load(open(f"{dossier}/parcelles_grille.geojson"))
cartes = {"ndvi_sec": "ndvi_seche_2026-08-08.tif", "ndvi_pluies": "ndvi_pluies_2026-04-10.tif",
          "ndmi_sec": "ndmi_seche_2026-08-08.tif", "ndmi_pluies": "ndmi_pluies_2026-04-10.tif"}

def moyennes(chemin):
    res = {}
    with rasterio.open(chemin) as src:
        arr = src.read(1)
        for ft in grille["features"]:
            geom = transform_geom("EPSG:4326", src.crs, ft["geometry"])
            dedans = geometry_mask([geom], out_shape=arr.shape, transform=src.transform, invert=True)
            v = arr[dedans]
            ok = ~np.isnan(v)
            res[ft["properties"]["nom"]] = (float(v[ok].mean()) if ok.any() else np.nan,
                                           float(ok.mean() * 100) if v.size else 0.0)
    return res

lignes = {}
for cle, fichier in cartes.items():
    for nom, (m, pct) in moyennes(f"{dossier}/{fichier}").items():
        lignes.setdefault(nom, {"nom": nom})
        lignes[nom][cle] = round(m, 3) if not np.isnan(m) else np.nan
        lignes[nom][cle + "_valides_pct"] = round(pct)

df = pd.DataFrame(list(lignes.values()))
df["delta_ndvi"] = (df["ndvi_pluies"] - df["ndvi_sec"]).round(3)
df["delta_ndmi"] = (df["ndmi_pluies"] - df["ndmi_sec"]).round(3)

# Alerte relative : les 10 % de carrés les plus faibles de la zone (carrés avec au moins 50 % de pixels valides)
fiable = df[[c for c in df.columns if c.endswith("_valides_pct")]].min(axis=1) >= 50
seuil_ndvi = df.loc[fiable, "ndvi_sec"].quantile(0.10)
seuil_ndmi = df.loc[fiable, "ndmi_sec"].quantile(0.10)
df["fiable"] = fiable
df["alerte"] = ""
df.loc[fiable & (df["ndvi_sec"] < seuil_ndvi), "alerte"] += "vigueur faible "
df.loc[fiable & (df["ndmi_sec"] < seuil_ndmi), "alerte"] += "humidité faible"
df["alerte"] = df["alerte"].str.strip()

df.to_csv(f"{dossier}/bulletin_grille.csv", index=False)

recs = {r["nom"]: r for r in json.loads(df.to_json(orient="records"))}
for ft in grille["features"]:
    ft["properties"] = recs[ft["properties"]["nom"]]
with open(f"{dossier}/parcelles_grille_indices.geojson", "w") as f:
    json.dump(grille, f)

print(len(df), "carrés |", int(fiable.sum()), "fiables |", int((df["alerte"] != "").sum()), "avec alerte")
print("Seuils (10 % les plus faibles) : NDVI", round(float(seuil_ndvi), 3), "| NDMI", round(float(seuil_ndmi), 3))
print(df.sort_values("ndvi_sec").head(8)[["nom", "ndvi_sec", "ndmi_sec", "delta_ndvi", "delta_ndmi", "alerte"]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as mp

with rasterio.open(f"{dossier}/ndvi_seche_2026-08-08.tif") as s:
    fond = s.read(1)

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(fond, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
for _, r in df.iterrows():
    li, co = map(int, r["nom"][1:].split("_"))
    alerte = bool(r["alerte"])
    ax.add_patch(mp.Rectangle((co * 100, li * 100), 100, 100, fill=False,
                              edgecolor="red" if alerte else "white",
                              linewidth=2 if alerte else 0.5))
    if alerte:
        ax.text(co * 100 + 5, li * 100 + 15, r["nom"], color="red", fontsize=8, weight="bold")
ax.set_title("NDVI saison sèche : carrés en alerte (rouge)")
ax.axis("off")
plt.show()